In [ ]:
from unstructured.partition.pdf import partition_pdf
# from unstructured.partition.auto import partition
from unstructured.documents.elements import Text,Element,FigureCaption,Image,Table
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
file_path = 'OLAP_and_OLTP.pdf'

raw_chunks = partition_pdf(
    filename=file_path,
    strategy="hi_res",
    infer_table_structure=True,
    extract_image_block_types=["figure", "table",'Image'],
    extract_image_block_to_payload=True,
    chunking_strategy=None,
)



In [ ]:
raw_chunks

In [ ]:
for idx, chunk in enumerate(raw_chunks):
    if isinstance(chunk, Image):
        print(f"Image {idx}")

In [ ]:
raw_chunks[42].to_dict()


In [ ]:
#you will observe that after image element we have figure caption element
raw_chunks[14].to_dict()

In [ ]:
all_images = []
for idx, chunk in enumerate(raw_chunks):
    if isinstance(chunk, Image):
        # the next element after the image will be figure caption
        if idx + 1 < len(raw_chunks) and isinstance(raw_chunks[idx + 1], FigureCaption):
            caption = raw_chunks[idx + 1].text
        else:
            caption = "No caption found"

        all_images.append({
            "index": idx,
            "caption": caption if caption else "No caption found",
            "image_text": chunk.text,
            "base64": chunk.metadata.image_base64
        })


In [ ]:
all_images

In [ ]:
import base64
from IPython.display import Image as IPImage, display
def display_image(image_dict):
    image_data = base64.b64decode(image_dict["base64"])
    display(IPImage(data=image_data, format='png'))
    print(f"Caption: {image_dict['caption']}")


for image_data in all_images:
    display_image(image_data)

### Image Captioning
very important process


In [ ]:
# In multi modal rag , we get image data , now we have to ingest image data also in vector database. We can use CLIP model to get image embeddings and then store those embeddings in vector database along with text chunks. During retrieval we can retrieve both text and image chunks based on query similarity.
# or knowledge base , so that we can use that knowledge base for multi modal question answering.
# so we can create the discriptions as text and then ingest those discriptions in vector database along with image embeddings. During retrieval we can retrieve both text and image chunks based on query similarity.

# we can do either local llm or extenal llm for image captioning.
# we require image captioning for creating the discriptions for images,tables , figures etc.

from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()
client = OpenAI()

def generate_image_discription(image_dict):
    prompt = f"""
    Generate and describe the image in detail. 
    The caption of image is {image_dict['caption']} and the image text is {image_dict['image_text']}
    Directly analyze the image and provide a detailed description without any additional text.
    
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{image_dict['base64']}"},
                    },
                ],
            }
        ],
    )
    return response.choices[0].message.content

generate_image_discription(all_images[0])

### NOW will do table parsing

In [1]:
table_data = []
for idx , element in enumerate(raw_chunks):
    if isinstance(element, Table):
        table_data.append({
            "table_as_html":element.metadata.text_as_html
        })
        

NameError: name 'raw_chunks' is not defined

In [ ]:
table_data[0]

In [ ]:
from IPython.display import HTML, display

def display_table(table_dict):
    html_content = table_dict["table_as_html"]
    display(HTML(html_content))

display_table(table_data[0])

In [ ]:
# generate table description using llm
def generate_table_description(table_html):
    prompt = f"""
    Generate and describe the table in detailed descriptiton of its contents , includding the strucutre, 
    key data points , notable treands or insights 
    The table is {table_html}
    Directly analyze the table and provide a detailed description without any additional text.
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )
    return response.choices[0].message.content

print(generate_table_description(table_data[0]['table_as_html']))


### Text parsing

In [ ]:
text_chunks = partition_pdf(
    filename=file_path,
    strategy="hi_res",
    chunking_strategy="by_title",
    max_characters=2000,
    combine_text_under_n_chars=500,
    new_after_n_chars=1500
)

In [ ]:
from unstructured.documents.elements import CompositeElement

for idx , chunk in enumerate(text_chunks):
    if isinstance(chunk, CompositeElement):
        print(f"chunk {idx}:{chunk.text[:50]}")